### Imports + Get as array:

In [1]:
import numpy as np
from pathlib import Path

cell_paths = np.load("metabolite_trajectories.npy", allow_pickle=True)
# trajectories = np.zeros((n_frames, n_metabolites, 3), dtype=np.int32)

# Going to have to reduce by factor of 10. I was told that this was at a 10 Angstrom per voxel, but we do reduce by a factor of 10.
print("Minimum: " + str(np.min(cell_paths)) + " Maximum: " + str(np.max(cell_paths))) 
# Diameter of ~20000 Angstom? syn3A cell is 400 nm diameter... I guess this was 5 Angstrom per voxel?

angstrom_per_voxel = 5

cell_paths = cell_paths / angstrom_per_voxel
print(cell_paths)

Minimum: 5.512 Maximum: 2137.5261
[[[142.00401  302.74802  273.358   ]
  [141.556    302.29602  273.34402 ]
  [141.578    301.87402  273.00803 ]
  ...
  [ 88.76601  296.50803  102.816   ]
  [138.98201  151.056    199.59401 ]
  [328.59003  125.51601  106.002   ]]

 [[141.58301  302.70303  273.70502 ]
  [141.56401  302.45303  273.52203 ]
  [141.694    302.23502  273.112   ]
  ...
  [ 89.253006 294.21503  102.384   ]
  [139.416    151.75601  198.91202 ]
  [329.71402  128.09601  106.996994]]

 [[141.702    302.3867   273.09467 ]
  [141.51933  302.19803  272.84467 ]
  [141.562    302.04733  272.39737 ]
  ...
  [ 89.55334  292.30334  102.47534 ]
  [139.548    151.92534  198.44    ]
  [329.55002  128.66867  107.72201 ]]

 ...

 [[132.8588   366.0916   307.83002 ]
  [132.99641  365.85004  307.51642 ]
  [133.2364   365.58844  307.3132  ]
  ...
  [ 65.6888   305.36603  413.13922 ]
  [103.49601  121.9968    75.328   ]
  [ 27.853601  85.174805  61.2128  ]]

 [[132.63521  366.1308   308.16962 ]
  [

In [2]:
# Important files:

# Position of marker
INITIAL_POSITION = [0, 0, 0]
# How far away from the marker we should be
MARKER_DISPLACEMENT = [0, 0, 0]

Print initial array:

In [5]:
files_dir = Path("files")
files_dir.mkdir(exist_ok=True)

n_metabolites = cell_paths.shape[1]
with open(files_dir / "summon.mcfunction", "w") as f:
    for i in range(n_metabolites):
        coord = cell_paths[0][i]

        # summon block_display -956 117 -879 {block_state:{Name:"tnt"},transformation:{scale:[2f,2f,2f],left_rotation:[0f,1f,0f,0f],right_rotation:[0f,1f,0f,0f],translation:[0f,0f,0f]},Tags:["ce.temp_oxygen","18"],teleport_duration:50}

        f.write(f"summon block_display ~{coord[0]} ~{coord[1]} ~{coord[2]} ")
        f.write("{block_state:{Name:\"purple_concrete\"},transformation:{scale:[1f,1f,1f],left_rotation:[0f,1f,0f,0f],right_rotation:[0f,1f,0f,0f],translation:[0f,0f,0f]},Tags:[\"ce.moving_martini2\",\"" + str(i) + "\"],teleport_duration:10}")
        f.write("\n")

### Write segments of array into code:

In [4]:
print(cell_paths.shape)
# execute as @n[type=minecraft:block_display,tag=ce.temp_oxygen,tag=18] at @s run data merge entity @s {transformation:{scale:[2f,2f,2f],left_rotation:[0f,1f,0f,0f],right_rotation:[0f,1f,0f,0f],translation:[5f,-37f,-23f]},interpolation_duration:100,start_interpolation:20}


(1121, 84, 3)


In [ ]:
smooth = 1
max_frames = 350

for frame_num in range(min(cell_paths.shape[0], max_frames)):
    if frame_num % smooth == 0:
        with open(files_dir / f"tick{frame_num // smooth}.mcfunction", "w") as f:
            for i in range(n_metabolites):
                coord = cell_paths[frame_num][i]

                # execute if entity @s[tag=1] run tp @s 0 0 0
                f.write(f"execute if entity @s[tag={i}] run return run tp @s {coord[0]} {coord[1]} {coord[2]}\n")
